# 21 — Temporal Split Design

**Objetivo:** congelar una validación verdaderamente temporal. No se usa `train_test_split` aleatorio.

In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m2_gate = paths.docs / "decisions" / "M2_GATE.md"
assert m2_gate.exists() and "M2 ✅ CLOSED" in m2_gate.read_text(encoding="utf-8"), (
    "M2 no está formalmente cerrado. No iniciar Gold."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")

{
    "gold_schema_version": GOLD_SPEC.schema_version,
    "target": GOLD_SPEC.target.name,
    "horizon_days": GOLD_SPEC.target.horizon_days,
    "event_scope": GOLD_SPEC.target.included_event_types,
    "duckdb_threads": GOLD_SPEC.runtime.duckdb_threads,
    "duckdb_memory_limit": GOLD_SPEC.runtime.duckdb_memory_limit,
    "m2_gate": str(m2_gate),
}

from argos.gold.splits import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION

## Política congelada

- Train: 2015–2023
- Validation: 2024
- Test: 2025
- 2026: elegible pero fuera del split principal, porque el año está parcialmente observado en el snapshot actual.

El test queda congelado para evaluación final y no se usa para feature engineering ni tuning.

In [ ]:
from argos.gold.splits import build_temporal_splits

result = build_temporal_splits(
    gold_dir=paths.gold,
    reports_root=paths.reports,
    manifest_dir=paths.root / "artifacts" / "manifests",
    gold_spec=GOLD_SPEC,
    temp_directory=paths.root / ".cache" / "duckdb",
    overwrite=True,
)

str(result.artifact.relative_to(paths.root))

In [ ]:
display(result.summary)
display(result.by_year)

In [ ]:
result.validation

In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
print("21 ✅ PASS" if result.ready_for_next_notebook else "21 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook, "Split temporal no supera sus checks bloqueantes."